# 加算器
量子コンピュータには、従来のコンピュータと同じ計算を実行できる機能があります。加算器の回路を見てみましょう。

## 今回学ぶこと
1. 量子ゲートを使った2進数加算器の実装
2. 量子的な重ね合わせを使うことで、1つの回路で複数の加算を同時に実現できること

## Blueqatのインストール
pipからBlueqatをインストールします。

In [1]:
!pip install git+https://github.com/blueqat/blueqatSDK

/Users/yuichirominato/.pyenv/shims/pip: line 8: /opt/homebrew/opt/pyenv/bin/pyenv: No such file or directory


## 2進数の足し算
加算器は、桁上げにCCXゲートを、足し算にCXゲートを使用します。ここでは、2進数の足し算 a+b = cd を行う量子回路を作ります。今回は、aとbの値に応じて4種類の足し算を実装します。それぞれの足し算は、

0+0 = 00 => 0000  
0+1 = 01 => 0101  
1+0 = 01 => 1001  
1+1 = 10 => 1110  

最初の2量子ビットが入力値a、bであり、後の2量子ビットが出力値c、dです。aとbの入力用の回路と、足し算用の回路を別々に作り、それを何度か使用します。データ入力には、0を1に反転させるためにXゲートを使います。

足し算の回路部分は以下のようになります。*は制御ビットです。

```
a ---*---*------- a
b ---*---|---*--- b
0 ---X---|---|--- c
0 -------X---X--- d
```

In [2]:
#import tools
from blueqat import Circuit
from collections import Counter as _Counter

# Compatibility note: the new blueqat SDK reports measurement bitstrings with
# qubit 0 as the right-most character (standard binary ordering), while this
# notebook (like the old blueqat 0.3.x) reads qubit 0 as the left-most
# character. Patch Circuit.run once so sampled bitstrings keep matching the
# qubit ordering used throughout this notebook (a, b, c, d ... read left to
# right as qubit 0, 1, 2, 3 ...).
if not getattr(Circuit, '_qubit0_leftmost_patched', False):
    _original_run = Circuit.run
    def _run_qubit0_leftmost(self, *args, **kwargs):
        result = _original_run(self, *args, **kwargs)
        if isinstance(result, _Counter):
            return _Counter({key[::-1]: count for key, count in result.items()})
        return result
    Circuit.run = _run_qubit0_leftmost
    Circuit._qubit0_leftmost_patched = True

#additon part
adder = Circuit().ccx[0,1,2].cx[0,3].cx[1,3]

In [3]:
#0+0
(Circuit() + adder).m[:].run(shots=100)

Counter({'0000': 100})

In [4]:
#0+1
(Circuit().x[1] + adder).m[:].run(shots=100)

Counter({'0101': 100})

In [5]:
#1+0
(Circuit().x[0] + adder).m[:].run(shots=100)

Counter({'1001': 100})

In [6]:
#1+1
(Circuit().x[0,1] + adder).m[:].run(shots=100)

Counter({'1110': 100})

このように、足し算ができました。

## 重ね合わせを使った足し算
次に、Xゲートで1つずつデータを入力する代わりに、Hゲートを使って足し算を行ってみましょう。

In [7]:
#Use H gate as input instead of X gate
(Circuit().h[0,1] + adder).m[:].run(shots=100)

Counter({'1110': 28, '1001': 28, '0101': 23, '0000': 21})

アダマールゲートを使うことで、4つの答えがそれぞれ約1/4ずつ得られました。このような汎用的な足し算回路を作れば、重ね合わせ状態を使って計算を行うことができます。

## エンタングルメントを使った足し算
次に、Hゲートの代わりに量子エンタングルメントを使って a+b=1 の足し算をしてみましょう。

In [8]:
#Make an entangled state of 01 and 10
(Circuit().h[0].cx[0,1].x[0] + adder).m[:].run(shots=100)

Counter({'1001': 53, '0101': 47})

01と10の入力値がエンタングルしているため、これら2つの足し算の結果がそれぞれ約半分ずつ出てきます。

--------

## 発展: 一般的な足し算
一般的な10進数同士の足し算を実装します。
$a, b$ の和を考えます。
$a, b$ は $a = a_n ... a_0$、$b = b_n ... b_0$ のように2進数で表せます。
(ここで n は大きい方の数を基準にします。)

回路は以下の通りです。

<img src="https://qiita-user-contents.imgix.net/https%3A%2F%2Fqiita-image-store.s3.ap-northeast-1.amazonaws.com%2F0%2F218694%2F489c8ea6-130c-d44b-4bae-4d86f64556c2.png?ixlib=rb-1.2.2&auto=format&gif-q=60&q=75&s=e67fbf71779fa7ebb443e93663c56dc4">

$c_i$ は桁上げビットです。
この回路は、桁上げ部分と合計部分の2つに分かれます。

まずこの2つの部分を考えます。

### 桁上げ

<img src="img/100_0.png" width="30%">

上から $c_i, a_i, b_i, c_{i+1}$ とし、$c_{i+1}$ が桁上げ部分を表します。

### 合計

<img src="img/100_1.png" width="30%">

上から $c_i, a_i, b_i$ とすると、3つの数の合計が $b_i$ に現れます。

### 実装

一般的な足し算は以下の手順で行います。

1. 桁上げ回路を使って桁上げビットを計算する。
2. CXゲートを使って桁上げの最後の部分を元に戻す。
3. 合計回路を使って各桁の合計を $b_n$ に出力する。
4. 逆桁上げ回路を使ってその桁の値を元に戻す。
5. 合計回路を使って各桁の合計を $b_i$ に出力する。
6. 4, 5を繰り返す。

$a+b$ は $b_{n+1} ... b_0$ に出力されます。
まず、桁上げ回路とその逆回路、合計回路を作ります。

In [9]:
from blueqat import Circuit

def carry(i):
    return Circuit().ccx[i+1,i+2,i+3].cx[i+1,i+2].ccx[i,i+2,i+3]

def carry_reverse(i):
    return Circuit().ccx[i,i+2,i+3].cx[i+1,i+2].ccx[i+1,i+2,i+3]

def sum(i):
    return Circuit().cx[i+1,i+2].cx[i,i+2]

10進数を2進数に変換する関数も作りましょう。

In [10]:
def tobinary(A):
    return bin(A)[2:]

tobinary(10)

'1010'

次に、2進数を回路にマッピングする関数を作ります。

In [11]:
def digits(a,b): 
     # change to the binary
     aa = tobinary(a)  
     bb = tobinary(b)  
     alen = len(aa)  
     blen = len(bb)  

     # get the value n
     maxlen = max(alen,blen) 
     if alen>blen: 
         bb = bb.zfill(alen) 
     elif blen>alen: 
         aa = aa.zfill(blen) 

     # mapping
     str = '' 
     for i in range(maxlen): 
         str += '0' + aa[maxlen-i-1] + bb[maxlen-i-1] 
     str += '0' 
     return str

digits(2,2)

'0000110'

回路の初期状態はすべて0であるため、マッピングされた値に合わせてXゲートを適用する必要があります。

In [12]:
def toX(a): 
     cir = Circuit(len(a)) 
     for i in range(len(a)): 
         if a[i] == "1": 
             cir += Circuit().x[i] 
     return cir

toX("101").m[:].run(shots=100)

Counter({'101': 100})

最後に、出力部分を考えます。
出力は2進数なので、10進数に変換します。

In [13]:
def todecimal(A):
    return int(str(A),2) 

todecimal(1001)

9

回路は $a_i, b_i, c_i$ が混ざったものを出力します。
$b_i$ だけを取り出します。

In [14]:
def getb(result): 
     str = result[-1] 
     digi = int((len(result)-1)/3) 
     for i in range(digi): 
         str += result[-2-i*3] 
     return todecimal(str)

getb("0000110")

2

### 一般的な回路

In [15]:
def plus(a,b): 
     # Mapping a binary number to a circuit
     qubits = len(digits(a,b)) 
     cir1 = toX(digits(a,b)) 
     digi = int((len(digits(a,b))-1)/3) 

     # first carry circuit
     cir2 = Circuit(qubits)     
     for i in range(digi): 
         cir2 += carry(i*3) 

     # last digit
     cir3 = Circuit(qubits).cx[-3,-2] + sum((digi-1)*3) 

     # Output the sum to bi
     cir4 = Circuit(qubits) 
     for i in range(digi-1): 
         cir4 += carry_reverse((digi-i-2)*3) 
         cir4 += sum((digi-i-2)*3)

     result = (cir1 + cir2 + cir3 + cir4).m[:].run(shots=1) 
     return getb(result.most_common()[0][0])

実際に計算してみましょう。

In [16]:
plus(2,2)

4

In [17]:
plus(13,15)

28

In [18]:
plus(70,90)

160

最後の計算には少し時間がかかりますが、一般的な加算器を実装することができました。

### 参考文献
V. Vedral, A. Barenco, A. Ekert, "Quantum Networks for Elementary Arithmetic Operations",  
(Submitted on 16 Nov 1995)  
https://arxiv.org/pdf/quant-ph/9511018.pdf

# 減算器
量子コンピュータには、従来のコンピュータと同じ計算を実行できる機能があります。減算器の回路を見てみましょう。

## 今回学ぶこと
1. 量子ゲートを使った2進数減算器の実装
2. 量子的な重ね合わせを使うことで、1つの回路で複数の減算を同時に実現できること

## 2進数の引き算
減算器は、符号の反転にCCXゲートを、引き算にCXゲートを使用します。ここでは、2進数の引き算 a-b = cd を行う量子回路を作ります。今回は、aとbの値に応じて4種類の引き算を実装します。それぞれの引き算は、

0-0 = 00 => 0000  
0-1 = 11 => 0111  
1-0 = 01 => 1001  
1-1 = 00 => 1100  

最初の2量子ビットが入力値a、bであり、後の2量子ビットが出力値c、dです。aとbの入力用の回路と、足し算用の回路を別々に作り、それを何度か使用します。データ入力には、0を1に反転させるためにXゲートを使います。

この回路部分は以下のようになります。*は制御ビットです。

```
a ---X---*---X---*------- a
b -------*-------|---*--- b
0 -------X-------|---|--- c
0 ---------------X---X--- d
```

In [19]:
#import tools
from blueqat import Circuit

#subtraction part
subtractor = Circuit().x[0].ccx[0,1,2].x[0].cx[0,3].cx[1,3]

In [20]:
#0-0
(Circuit() + subtractor).m[:].run(shots=100)

Counter({'0000': 100})

In [21]:
#0-1
(Circuit().x[1] + subtractor).m[:].run(shots=100)

Counter({'0111': 100})

In [22]:
#1-0
(Circuit().x[0] + subtractor).m[:].run(shots=100)

Counter({'1001': 100})

In [23]:
#1-1
(Circuit().x[0,1] + subtractor).m[:].run(shots=100)

Counter({'1100': 100})

このように、計算ができました。

## 重ね合わせを使った引き算
次に、Xゲートで1つずつデータを入力する代わりに、Hゲートを使って引き算を行ってみましょう。

In [24]:
#Use H gate as input instead of X gate
(Circuit().h[0,1] + subtractor).m[:].run(shots=100)

Counter({'1100': 29, '0000': 26, '0111': 23, '1001': 22})

アダマールゲートを使うことで、4つの答えがそれぞれ約1/4ずつ得られました。このような汎用的な引き算回路を作れば、重ね合わせ状態を使って計算を行うことができます。

## エンタングルメントを使った引き算
次に、Hゲートの代わりに量子エンタングルメントを使って a-b=0 の引き算をしてみましょう。

In [25]:
#Make an entangled state of 00 and 11
(Circuit().h[0].cx[0,1] + subtractor).m[:].run(shots=100)

Counter({'1100': 55, '0000': 45})

00と11の入力値がエンタングルしているため、これら2つの足し算の結果がそれぞれ約半分ずつ出てきます。

--------

## 発展: 一般的な引き算
一般的な10進数同士の引き算を実装します。
引き算回路は、足し算回路を逆にすることで実装できます。

<img src="https://qiita-user-contents.imgix.net/https%3A%2F%2Fqiita-image-store.s3.ap-northeast-1.amazonaws.com%2F0%2F218694%2F489c8ea6-130c-d44b-4bae-4d86f64556c2.png?ixlib=rb-1.2.2&auto=format&gif-q=60&q=75&s=e67fbf71779fa7ebb443e93663c56dc4">

回路を右から考えます。$a, a+b$ を入力して $b$ を出力します。
$a = a_n ... a_0$、$b = b_n ... b_0$ のように2進数で表せます。
(ここで n は大きい方の数を基準にします。)

$c_i$ は桁下げです。

### 桁下げ

<img src="https://qiita-user-contents.imgix.net/https%3A%2F%2Fqiita-image-store.s3.ap-northeast-1.amazonaws.com%2F0%2F218694%2F00361854-e782-0da0-3386-f59f668a1ada.png?ixlib=rb-1.2.2&auto=format&gif-q=60&q=75&s=58d1d41b0893b16bfb24bfd546328ff2" width="60%">

回路の最初の部分で、これらを組み合わせて桁下げを求めます。

### オーバーフロービット
判定は $b_{n+1}$ によって与えられます。これはオーバーフロービットと呼ばれます。

$a < a+b$ のとき $b_{n+1}$ = 0
$a > a+b$ のとき $b_{n+1}$ = 1

$a > a+b$ の場合、$2^{n+1} - b$ が出力されます。

### 実装

一般的な引き算は以下の手順で行います。

1. 逆回路と合計回路を使って桁上げビットを計算する。
2. オーバーフローを確認して $b_{n+1}$ に格納する。
3. 桁上げ回路を使って差を計算する。

$b$ は $b_n ... b_0$ に出力され、$b_{n+1}$ がオーバーフロービットです。
実装の準備をしましょう。これは足し算の回路と同じです。

In [26]:
from blueqat import Circuit

def carry(i):
    return Circuit().ccx[i+1,i+2,i+3].cx[i+1,i+2].ccx[i,i+2,i+3]

def carry_reverse(i):
    return Circuit().ccx[i,i+2,i+3].cx[i+1,i+2].ccx[i+1,i+2,i+3]

def sum_reverse(a):
    return Circuit().cx[a,a+2].cx[a+1,a+2]

def tobinary(A):
    return bin(A)[2:]

def digits(a,b): 
     # change to the binary
     aa = tobinary(a)  
     bb = tobinary(b)  
     alen = len(aa)  
     blen = len(bb)  

     # get the value n
     maxlen = max(alen,blen) 
     if alen>blen: 
         bb = bb.zfill(alen) 
     elif blen>alen: 
         aa = aa.zfill(blen) 

     # mapping
     str = '' 
     for i in range(maxlen): 
         str += '0' + aa[maxlen-i-1] + bb[maxlen-i-1] 
     str += '0' 
     return str

def toX(a): 
     cir = Circuit(len(a)) 
     for i in range(len(a)): 
         if a[i] == "1": 
             cir += Circuit().x[i] 
     return cir

def todecimal(A):
    return int(str(A),2) 

def getb(result): 
     str = result[-1] 
     digi = int((len(result)-1)/3) 
     for i in range(digi): 
         str += result[-2-i*3] 
     return todecimal(str)

In [27]:
def minus(a,ab): 
     # swap a for ab
     c = ab
     ab = a
     a = c

     # Mapping a binary number to a circuit
     qubits = len(digits(a,ab)) 
     cir1 = toX(digits(a,ab)) 
     digi = int((len(digits(a,ab))-1)/3) 

     # first carry circuit and sum reverse circuit
     cir4 = Circuit(qubits) 
     for i in range(digi-1): 
         cir4 += sum_reverse(i*3)
         cir4 += carry(i*3) 

     # last digit
     cir3 = sum_reverse((digi-1)*3) + Circuit(qubits).cx[-3,-2] 

     # Output the subtraction to bi
     cir2 = Circuit(qubits)     
     for i in range(digi): 
         cir2 += carry_reverse((digi-1-i)*3)

     result = (cir1 + cir4 + cir3 + cir2).m[:].run(shots=1) 
     return getb(result.most_common()[0][0])

実際に計算してみましょう。

In [28]:
minus(8,2)

6

In [29]:
minus(4,2)

2

In [30]:
minus(50,24)

26

計算できました。ちなみに、$a > a + b$ の場合について、

In [31]:
minus(2,4)

14

これも計算できています。

### 参考文献
V. Vedral, A. Barenco, A. Ekert, "Quantum Networks for Elementary Arithmetic Operations",  
(Submitted on 16 Nov 1995)  
https://arxiv.org/pdf/quant-ph/9511018.pdf

# 乗算器
ここでは、加算器の回路を使った基本的な乗算器の回路を見ていきます。

## 2進数の乗算器について
基本的な乗算器の回路は、2つの2進数を掛け合わせるCCXゲートから構成されます。

$$0*0=0\\
0*1=0\\
1*0=0\\
1*1=1$$

そして、各桁を合計します。

## 例
以下を解いてみましょう。

$$1*2 = ?$$

この計算を2進数で表すと

$$01*10 = 0010$$


```
    01
*  10
-------
    00
 01
-------
    0
 0
-------
 0010
  ```
  
ここで、aとしてa0,a2を、bとしてb0,b2を用意します。
aとbを掛けた後の処理をc0,c21,c22,c4とします。
z4,z8を桁上げとします。
そしてx0からx8を結果とします。

<img src="img/010_basic_multi01.png?raw=1">

In [32]:
from blueqat import Circuit

C1 = Circuit().ccx[0,1,2].ccx[1,3,5].ccx[0,4,6].ccx[3,4,7].ccx[5,6,8].ccx[7,8,9].cx[2,10].cx[5,11].cx[6,11].cx[7,12].cx[8,12].cx[9,13] 

#00 * 00 = 0000
(Circuit() + C1).m[:].run(shots=100)

Counter({'00000000000000': 100})

In [33]:
#01 * 01 = 0001
(Circuit().x[0].x[1] + C1).m[:].run(shots=100)

Counter({'11100000001000': 100})

In [34]:
#10 * 01 = 0010
(Circuit().x[3].x[1] + C1).m[:].run(shots=100)

Counter({'01010100000100': 100})

In [35]:
#11 * 10 = 0110
(Circuit().x[0].x[3].x[4] + C1).m[:].run(shots=100)

Counter({'10011011000110': 100})

In [36]:
#10 * 11 = 0110
(Circuit().x[1].x[3].x[4] + C1).m[:].run(shots=100)

Counter({'01011101000110': 100})

In [37]:
#11 * 11 = 1001
(Circuit().x[0].x[1].x[3].x[4] + C1).m[:].run(shots=100) 

Counter({'11111111111001': 100})

これで乗算器の結果が分かりました。最後の4量子ビットの並びが結果を表します。

# 剰余
今回は、一般的な足し算と引き算の回路を使って剰余を求めます。

回路は以下の通りです。

<img src="https://qiita-user-contents.imgix.net/https%3A%2F%2Fqiita-image-store.s3.ap-northeast-1.amazonaws.com%2F0%2F218694%2Ff161e722-f727-b6fb-d02b-24e40b5b63b1.png?ixlib=rb-1.2.2&auto=format&gif-q=60&q=75&s=da298eace648814b5ccfe4affa9301ca">

$a,b$ は $0 < a,b < N$ です。最後のビットは一時ビットと呼ばれます。オーバーフローを確認するためのものです。

## 実装手順
$a+b\mod N$ を求めるには、$a+b$ と $N$ を比較する必要があります。

$a+b>N$ のとき、  
$a+b-N = (a+b) \mod N$

$a+b<N$ のとき、 
$a+b = (a+b) \mod N$

これを量子回路で実装します。

$a+b>N$ のとき、

<img src="https://qiita-user-contents.imgix.net/https%3A%2F%2Fqiita-image-store.s3.ap-northeast-1.amazonaws.com%2F0%2F218694%2F5c97b6fd-425b-c42b-057a-68d9cb47e457.jpeg?ixlib=rb-1.2.2&auto=format&gif-q=60&q=75&s=3711f239a543dc4aa175f4636adde888">

$a+b<N$ のとき、

<img src="https://qiita-user-contents.imgix.net/https%3A%2F%2Fqiita-image-store.s3.ap-northeast-1.amazonaws.com%2F0%2F218694%2Fcb7bc1bf-e713-f7d4-9cdc-4bc47bb6eba8.jpeg?ixlib=rb-1.2.2&auto=format&gif-q=60&q=75&s=425f2f86ef9d5b5c27b1c1f1ddfab936">

余った量子ビットと最上位の量子ビットの値を使用します。

## 例
簡単な例を見てみましょう。

$3 + 5 < 7$ のとき、  
$(3 + 5) \mod 7 = 1$

$3 + 5 > 11$ のとき、    
$(3 + 5) \mod 11 = 8$

これを量子回路で実装しようとしています。

### 実装
実装の準備をしましょう。これは足し算の回路と引き算の回路と同じです。

一時ビットとNを対応させるために、上記の回路にもう1つNを追加します。
初期状態は以下の通りです。

```
c0 --
a0 --
b0 --
c1 --
..
n0 --
n1 --
..
t  --
n0 --
n1 --
..
```

また、各桁は $N$ です。

In [38]:
from blueqat import Circuit

def carry(a):
    return Circuit().ccx[a+1,a+2,a+3].cx[a+1,a+2].ccx[a,a+2,a+3]

def carry_reverse(a):
    return Circuit().ccx[a,a+2,a+3].cx[a+1,a+2].ccx[a+1,a+2,a+3]

def sum(a):
    return Circuit().cx[a+1,a+2].cx[a,a+2] 

def sum_reverse(a):
    return Circuit().cx[a,a+2].cx[a+1,a+2]

def tobinary(A):
    return bin(A)[2:] 

# get the digit number
def digits2(a,b,n): 
     aa = tobinary(a)  
     bb = tobinary(b)  
     nn = tobinary(n)  

     nlen = len(nn)  
     aa = aa.zfill(nlen) 
     bb = bb.zfill(nlen) 

     str = '' 
     for i in range(nlen): 
         str += '0' + aa[nlen-i-1] + bb[nlen-i-1] 
     str += '0' 

     for i in range(nlen): 
         str += nn[nlen-i-1]  

     str += '0'

     for i in range(nlen): 
        str += nn[nlen-i-1]

     return str

# initial states
def toX(a): 
     cir = Circuit(len(a)) 
     for i in range(len(a)): 
         if a[i] == "1": 
             cir += Circuit().x[i] 
     return cir

# adder
def plus(a,b,n): 
     qubits = len(digits2(a,b,n))
     digi = len(tobinary(n))

     cir2 = Circuit(qubits)     
     for i in range(digi): 
         cir2 += carry(i*3) 

     cir3 = Circuit(qubits).cx[(digi-1)*3+1,(digi-1)*3+2] + sum((digi-1)*3)

     cir4 = Circuit(qubits) 
     for i in range(digi-1): 
         cir4 += carry_reverse((digi-i-2)*3) 
         cir4 += sum((digi-i-2)*3)

     cir_plus = cir2 + cir3 + cir4
     return cir_plus

# adder_inverse
def minus(a,ab,n): 
     qubits = len(digits2(a,ab,n))
     digi = len(tobinary(n))

     cir4 = Circuit(qubits) 
     for i in range(digi-1): 
         cir4 += sum_reverse(i*3)
         cir4 += carry(i*3) 

     cir3 = sum_reverse((digi-1)*3) + Circuit(qubits).cx[(digi-1)*3+1,(digi-1)*3+2] 

     cir2 = Circuit(qubits)     
     for i in range(digi): 
         cir2 += carry_reverse((digi-1-i)*3) 

     cir_minus = cir4 + cir3 + cir2
     return cir_minus

# swap a and N
def swap(n):
    digi = len(tobinary(n))
    cir = Circuit(5*digi+2)
    for i in range(digi):
        cir += Circuit(5*digi+2).swap[3*i+1,3*digi+1+i]
    return cir

# to decimal number
def todecimal(A):
    return int(str(A),2) 

# get b from the circuit
def getb(result,n): 
     str = ""
     digi = len(tobinary(n)) 
     for i in range(digi): 
         str += result[3*(digi-i)-1]
     return todecimal(str)

### 一般的な回路

In [39]:
def adder_mod(a,b,n):
    digi = len(tobinary(n))

    # first part
    part1 = toX(digits2(a,b,n)) + plus(a,b,n) + swap(n) + minus(a,b,n)

    # set the overflow to a temporary bit
    part2 = Circuit(5*digi+2).x[digi*3].cx[digi*3,digi*4+1].x[digi*3]

    # returns N from a temporary bit.
    part3 = Circuit(5*digi+2)
    for i in range(digi):
        part3 += Circuit(5*digi+2).ccx[4*digi+2+i,4*digi+1,3*i+1]

    # last part
    part4 = minus(a,b,n)+Circuit(5*digi+2).cx[digi*3,digi*4+1]+plus(a,b,n)
    
    result = (part1+part2+part3+plus(a,b,n)+part3+swap(n)+part4).m[:].run(shots=1)
    return getb(result.most_common()[0][0],n)

実際に計算してみましょう。

In [40]:
adder_mod(4,3,5)

2

In [41]:
adder_mod(4,4,5)

3

In [42]:
adder_mod(1,5,6)

0

計算できました。